# Review Cleaning

Turns the raw JSONL into typed parquet, one file per game: keeps the fields the study uses, drops duplicates, hashes the Steam ids, and computes post-review persistence.

# Setup and Paths

One parquet per game, because the full review corpus with text does not fit in memory.

In [1]:
import json
import hashlib
import datetime as dt
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# same paths as the collection notebook
project_root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
raw_reviews_dir=project_root/'data'/'raw'/'reviews'
checkpoint_dir=project_root/'data'/'raw'/'_checkpoints'
clean_reviews_dir=project_root/'data'/'processed'/'interim'/'reviews'
clean_reviews_dir.mkdir(parents=True,exist_ok=True)

rejects_file_path=project_root/'data'/'processed'/'interim'/'clean_rejects.csv'
tables_dir=project_root/'data'/'processed'/'tables'
tables_dir.mkdir(parents=True,exist_ok=True)

print('reading raw jsonl from :',raw_reviews_dir)
print('writing clean parquet to:',clean_reviews_dir)
print()
print('raw games on disk right now:',len(list(raw_reviews_dir.glob('*.jsonl'))))

reading raw jsonl from : d:\ea-health-scoring\data\raw\reviews
writing clean parquet to: d:\ea-health-scoring\data\processed\interim\reviews

raw games on disk right now: 1793


# Deciding what a clean review row looks like

Only the fields the research question uses are carried forward.

In [2]:
# hash the steam id - stable, not reversible
def hash_steam_id(steam_id):
    return hashlib.sha256(str(steam_id).encode()).hexdigest()[:16]


# steam sends some numbers as strings, so coerce types once here
def to_int(value,default=None):
    try:
        return int(value)
    except (TypeError,ValueError):
        return default


def flatten_review(raw_review):
    """turn one raw steam review dict into the flat row i actually want to keep"""
    author=raw_review.get('author',{})

    return {'review_id':raw_review.get('recommendationid'),
            'appid':to_int(raw_review.get('_appid')),
            'author_hash':hash_steam_id(author.get('steamid')),
            'playtime_at_review':to_int(author.get('playtime_at_review')),
            'playtime_forever':to_int(author.get('playtime_forever')),
            'playtime_last_two_weeks':to_int(author.get('playtime_last_two_weeks'),0),
            'author_games_owned':to_int(author.get('num_games_owned'),0),
            'author_review_count':to_int(author.get('num_reviews'),0),
            'written_during_early_access':bool(raw_review.get('written_during_early_access')),
            'timestamp_created':to_int(raw_review.get('timestamp_created')),
            'timestamp_updated':to_int(raw_review.get('timestamp_updated')),
            'voted_up':bool(raw_review.get('voted_up')),
            'votes_up':to_int(raw_review.get('votes_up'),0),
            'votes_funny':to_int(raw_review.get('votes_funny'),0),
            'comment_count':to_int(raw_review.get('comment_count'),0),
            'weighted_vote_score':float(raw_review.get('weighted_vote_score') or 0),
            'review_text_raw':raw_review.get('review') or '',
            'language':raw_review.get('language'),
            'steam_purchase':bool(raw_review.get('steam_purchase')),
            'received_for_free':bool(raw_review.get('received_for_free')),
            'refunded':bool(raw_review.get('refunded')),
            'app_release_date':to_int(raw_review.get('app_release_date')),
            'window':raw_review.get('_window')}


print('flattening one real row to check the mapping holds')

# the collector may still be running, so this is normal
raw_files_now=sorted(raw_reviews_dir.glob('*.jsonl'))
if not raw_files_now:
    print('no raw jsonl on disk yet - the collector has not written anything i can read.')
    print('nothing below this point will have data to work on. re run once collection starts')
else:
    sample_row=flatten_review(json.loads(raw_files_now[0].open(encoding='utf-8').readline()))
    for field,value in sample_row.items():
        shown=str(value)[:60]
        print(f'{field:30s} {shown}')

flattening one real row to check the mapping holds
review_id                      231664744
appid                          1000030
author_hash                    98b4e2d2121fbed2
playtime_at_review             22938
playtime_forever               23285
playtime_last_two_weeks        2596
author_games_owned             0
author_review_count            3
written_during_early_access    False
timestamp_created              1785417069
timestamp_updated              1785417069
voted_up                       True
votes_up                       0
votes_funny                    0
comment_count                  0
weighted_vote_score            0.5
review_text_raw                I love this game. It's silly, it's pretty, and it is a real 
language                       english
steam_purchase                 True
received_for_free              False
refunded                       False
app_release_date               1580324436
window                         whole_history


# English only, and cleaning the text properly

English rows only, with markup, urls and non-latin characters stripped into a separate column. The per-game English share is counted *before* the filter runs, because it is a covariate.

In [3]:
import re
import unicodedata

# strip steam bbcode by tag shape, before the censored token is inserted
steam_markup_pattern=re.compile(r'\[/?[a-zA-Z][a-zA-Z0-9]*(?:=[^\]]*)?\]')
url_pattern=re.compile(r'(?:https?://|www\.)\S+')
# mop up any bare url scheme left by spaced-out links
bare_scheme_pattern=re.compile(r'\b(?:https?://|www\.)')
censored_pattern=re.compile(r'\u2665{2,}')
control_character_pattern=re.compile(r'[\u200b-\u200f\u202a-\u202e\ufeff\x00-\x08\x0b\x0c\x0e-\x1f]')

# keep latin, digits, whitespace, punctuation; brackets for the mask
allowed_character_pattern=re.compile(r"[^A-Za-z0-9\u00c0-\u024f\s!\"#$%&'()*+,\-./:;<=>?@\[\]^_`{|}~]")

latin_letter_pattern=re.compile(r'[A-Za-z\u00c0-\u024f]')
any_letter_pattern=re.compile(r'[^\W\d_]')


def keep_any_mask_inside(match):
    """markup goes, but any censorship marker sitting inside the tag stays"""
    # a tag attribute can contain a mask, so re-emit any markers the tag carried
    marker_count=match.group().count('CENSOREDMASKTOKEN')
    return ' CENSOREDMASKTOKEN '*marker_count if marker_count else ' '


def clean_review_text(text):
    """markup, urls and non language characters out. profanity mask kept as a token"""
    # tokenise the mask first or the url pattern eats it
    # the placeholder has no brackets so the markup filter cannot eat it
    text=censored_pattern.sub(' CENSOREDMASKTOKEN ',text)

    text=steam_markup_pattern.sub(keep_any_mask_inside,text)
    text=url_pattern.sub(' ',text)
    text=bare_scheme_pattern.sub(' ',text)
    text=control_character_pattern.sub('',text)
    text=allowed_character_pattern.sub(' ',text)

    text=text.replace('CENSOREDMASKTOKEN','[censored]')
    return re.sub(r'\s+',' ',text).strip()


def latin_script_share(text):
    """what fraction of the letters are latin. catches rows whose language field lies"""
    letters=any_letter_pattern.findall(text)
    if not letters:
        return 1.0
    return len(latin_letter_pattern.findall(text))/len(letters)


print('checking the cleaner on the cases that actually matter')
cleaning_examples=[
    ('[h1]Great game[/h1] see [url=http://x.com]this[/url]','markup and a url'),
    ('this game is \u2665\u2665\u2665\u2665 broken','the profanity mask'),
    ('good game \U0001f600\U0001f602 \u597d\u73a9','emoji and chinese'),
    ('THIS IS AWFUL!!! why?!','casing and punctuation must survive'),
    ('\u597d\u73a9\u7684\u6e38\u620f','a review with no latin at all'),
]
for original,note in cleaning_examples:
    print(f'  {note}')
    print(f'    in  : {original}')
    print(f'    out : {clean_review_text(original)}')
    print(f'    latin share {latin_script_share(original):.2f}')

# assert the carve-outs, so a later regex edit cannot quietly undo them
assert clean_review_text('this game is \u2665\u2665\u2665\u2665 broken')=='this game is [censored] broken'
assert clean_review_text('THIS IS AWFUL!!! why?!')=='THIS IS AWFUL!!! why?!'
assert '[censored]' in clean_review_text('[b]bold[/b] \u2665\u2665\u2665\u2665')
assert latin_script_share('\u597d\u73a9\u7684\u6e38\u620f')==0.0
# the mask must survive inside a url
# the cost is url debris left behind, which is noise the scorers ignore
url_with_mask=clean_review_text('see https://tenor.com/view/\u2665\u2665\u2665\u2665-gif-2603')
assert '[censored]' in url_with_mask,'the mask was eaten by the url pattern again'
assert 'https://' not in url_with_mask,'the link survived'
# and inside a bbcode attribute, which is a separate path
assert '[censored]' in clean_review_text('[quote=This \u2665\u2665\u2665\u2665 Game]hello[/quote]'),    'the mask was eaten by the markup pattern'
print('carve outs hold: mask kept as a token, casing and punctuation untouched')

checking the cleaner on the cases that actually matter
  markup and a url
    in  : [h1]Great game[/h1] see [url=http://x.com]this[/url]
    out : Great game see this
    latin share 1.00
  the profanity mask
    in  : this game is ♥♥♥♥ broken
    out : this game is [censored] broken
    latin share 1.00
  emoji and chinese
    in  : good game 😀😂 好玩
    out : good game
    latin share 0.80
  casing and punctuation must survive
    in  : THIS IS AWFUL!!! why?!
    out : THIS IS AWFUL!!! why?!
    latin share 1.00
  a review with no latin at all
    in  : 好玩的游戏
    out : 
    latin share 0.00
carve outs hold: mask kept as a token, casing and punctuation untouched


# Cleaning one game

Rows are dropped for three reasons only, and every drop is counted for the report.

In [4]:
def clean_one_game(appid):
    """read one raw jsonl file and write one typed parquet file. returns the reject counts"""
    raw_file_path=raw_reviews_dir/f'{appid}.jsonl'
    clean_file_path=clean_reviews_dir/f'{appid}.parquet'

    rejects={'appid':appid,'no_review_id':0,'duplicate':0,'no_steam_id':0,
             'not_english':0,'not_latin_script':0,'empty_after_cleaning':0,'kept':0}
    seen_review_ids=set()
    kept_rows=[]

    # line by line, the big games are 30 mb each
    with raw_file_path.open(encoding='utf-8') as raw_file:
        for line in raw_file:
            raw_review=json.loads(line)

            review_id=raw_review.get('recommendationid')
            if not review_id:
                rejects['no_review_id']+=1
                continue
            if review_id in seen_review_ids:
                rejects['duplicate']+=1
                continue
            if not raw_review.get('author',{}).get('steamid'):
                rejects['no_steam_id']+=1
                continue

            seen_review_ids.add(review_id)
            kept_rows.append(flatten_review(raw_review))

    df_clean=pd.DataFrame(kept_rows)
    if df_clean.empty:
        return rejects

    # denominator, counted before any filtering
    # after filtering every game reads as 100% english, so count now
    is_english=df_clean['language']=='english'
    rejects['n_reviews_all_languages']=len(df_clean)
    rejects['n_english']=int(is_english.sum())
    rejects['english_share']=round(100*is_english.mean(),2)
    rejects['n_early_access_all_languages']=int(df_clean['written_during_early_access'].sum())
    rejects['top_other_language']=(df_clean.loc[~is_english,'language'].value_counts().idxmax()
                                   if (~is_english).any() else None)

    # filter
    rejects['not_english']=int((~is_english).sum())
    df_clean=df_clean[is_english].copy()

    # second check, claims english but the text is not
    # only runs on rows already claiming english
    if not df_clean.empty:
        latin_share=df_clean['review_text_raw'].map(latin_script_share)
        rejects['not_latin_script']=int((latin_share<0.70).sum())
        df_clean=df_clean[latin_share>=0.70].copy()

    if df_clean.empty:
        return rejects

    # clean into a new column, never over the original
    df_clean['review_text_clean']=df_clean['review_text_raw'].map(clean_review_text)

    # emoji-only or markup-only reviews have nothing left to score
    cleaned_to_nothing=df_clean['review_text_clean'].str.strip()==''
    rejects['empty_after_cleaning']=int(cleaned_to_nothing.sum())
    df_clean=df_clean[~cleaned_to_nothing].copy()

    rejects['kept']=len(df_clean)
    if df_clean.empty:
        return rejects

    # real dates, needed by the velocity and trajectory features
    df_clean['review_date']=pd.to_datetime(df_clean['timestamp_created'],unit='s')
    df_clean['updated_date']=pd.to_datetime(df_clean['timestamp_updated'],unit='s')
    df_clean['release_date']=pd.to_datetime(df_clean['app_release_date'],unit='s')
    df_clean['days_since_release']=(df_clean['review_date']-df_clean['release_date']).dt.days

    # steam recomputes the EA flag on edit, so edits can be mislabelled
    df_clean['was_edited']=df_clean['timestamp_updated']>df_clean['timestamp_created']

    # the metric this project rests on: playtime after the review
    df_clean['post_review_persistence']=df_clean['playtime_forever']-df_clean['playtime_at_review']

    # quality flags kept instead of dropping rows
    df_clean['has_playtime']=df_clean['playtime_at_review'].notna()&(df_clean['playtime_at_review']>0)
    df_clean['persistence_is_negative']=df_clean['post_review_persistence']<0

    df_clean.to_parquet(clean_file_path,index=False)
    return rejects


print('cleaning function is defined, testing it on one game before i run the lot')
raw_files_now=sorted(raw_reviews_dir.glob('*.jsonl'))
if raw_files_now:
    print(clean_one_game(int(raw_files_now[0].stem)))
else:
    print('no raw games to test against yet')

cleaning function is defined, testing it on one game before i run the lot
{'appid': 1000030, 'no_review_id': 0, 'duplicate': 0, 'no_steam_id': 0, 'not_english': 553, 'not_latin_script': 4, 'empty_after_cleaning': 10, 'kept': 1735, 'n_reviews_all_languages': 2302, 'n_english': 1749, 'english_share': np.float64(75.98), 'n_early_access_all_languages': 760, 'top_other_language': 'schinese'}


# Running it over everything on disk

Incremental: a game whose parquet is newer than its raw file is skipped, so this can be re-run while collection is still going.

In [5]:
def needs_cleaning(raw_file_path):
    """skip a game whose clean parquet is already newer than its raw jsonl"""
    clean_file_path=clean_reviews_dir/f'{raw_file_path.stem}.parquet'
    if not clean_file_path.exists():
        return True
    return clean_file_path.stat().st_mtime<raw_file_path.stat().st_mtime


raw_files=sorted(raw_reviews_dir.glob('*.jsonl'))

# a clean file with no raw file would never be revisited, so sweep it here
live_appids={f.stem for f in raw_files}
orphans=[f for f in clean_reviews_dir.glob('*.parquet') if f.stem not in live_appids]
for orphan_file in orphans:
    orphan_file.unlink()
if orphans:
    print(f'removed {len(orphans)} orphaned clean files with no raw counterpart: '
          f'{[o.stem for o in orphans][:10]}')

pending_files=[f for f in raw_files if needs_cleaning(f)]

print('CLEANING RUN')
print(f'raw games on disk : {len(raw_files)}')
print(f'already clean     : {len(raw_files)-len(pending_files)}')
print(f'to clean now      : {len(pending_files)}')

reject_rows=[]
for position,raw_file_path in enumerate(pending_files,start=1):
    appid=int(raw_file_path.stem)
    rejects=clean_one_game(appid)
    reject_rows.append(rejects)
    print(f'[{position}/{len(pending_files)}] app {appid}: kept {rejects["kept"]:,} of '
          f'{rejects.get("n_reviews_all_languages",0):,} '
          f'| english {rejects.get("english_share",0)}% '
          f'| dropped: {rejects["not_english"]:,} non-english, '
          f'{rejects["not_latin_script"]:,} non-latin, '
          f'{rejects["empty_after_cleaning"]:,} empty after cleaning, '
          f'{rejects["duplicate"]:,} dupes')

# reject counts go in the report, so write them to disk
if reject_rows:
    df_rejects=pd.DataFrame(reject_rows)
    if rejects_file_path.exists():
        df_rejects_old=pd.read_csv(rejects_file_path)
        df_rejects=pd.concat([df_rejects_old[~df_rejects_old['appid'].isin(df_rejects['appid'])],df_rejects])
    df_rejects.to_csv(rejects_file_path,index=False)
    print()
    print('reject counts written to',rejects_file_path.name)

CLEANING RUN
raw games on disk : 1793
already clean     : 1699
to clean now      : 94
[1/94] app 1033410: kept 196 of 272 | english 73.9% | dropped: 71 non-english, 1 non-latin, 4 empty after cleaning, 0 dupes
[2/94] app 1066890: kept 609 of 1,171 | english 52.18% | dropped: 560 non-english, 0 non-latin, 2 empty after cleaning, 0 dupes
[3/94] app 108600: kept 1,496 of 4,400 | english 34.05% | dropped: 2,902 non-english, 2 non-latin, 0 empty after cleaning, 0 dupes
[4/94] app 1120480: kept 1,281 of 1,883 | english 68.4% | dropped: 595 non-english, 0 non-latin, 7 empty after cleaning, 0 dupes
[5/94] app 1134100: kept 1,773 of 3,904 | english 45.57% | dropped: 2,125 non-english, 5 non-latin, 1 empty after cleaning, 0 dupes
[6/94] app 1144200: kept 3,239 of 5,900 | english 55.05% | dropped: 2,652 non-english, 8 non-latin, 1 empty after cleaning, 0 dupes
[7/94] app 1168870: kept 216 of 780 | english 27.82% | dropped: 563 non-english, 0 non-latin, 1 empty after cleaning, 0 dupes
[8/94] app 1

# Verifying the assumptions the whole design rests on

Four things the design assumes about Steam reviews, checked against the data rather than trusted.

In [6]:
# one row per game, never concatenate review-level tables
summary_rows=[]

summary_columns=['written_during_early_access','review_date','release_date','has_playtime',
                 'persistence_is_negative','post_review_persistence','author_hash','was_edited']

for clean_file_path in sorted(clean_reviews_dir.glob('*.parquet')):
    # columns only, the text is the bulk of the corpus
    df_game=pd.read_parquet(clean_file_path,columns=summary_columns)
    if df_game.empty:
        continue

    df_ea=df_game[df_game['written_during_early_access']]
    df_post=df_game[~df_game['written_during_early_access']]
    df_persistence=df_ea[df_ea['has_playtime']]

    summary_rows.append({
        'appid':int(clean_file_path.stem),
        'release_date':df_game['release_date'].max().date(),
        'n_reviews':len(df_game),
        'n_early_access':len(df_ea),
        'n_post_launch':len(df_post),
        'pct_with_playtime':round(100*df_game['has_playtime'].mean(),1),
        'pct_persistence_negative':round(100*df_game['persistence_is_negative'].mean(),1),
        'median_persistence_hours':round(df_persistence['post_review_persistence'].median()/60,1) if len(df_persistence) else None,
        'n_ea_authors':df_ea['author_hash'].nunique(),
        'n_overlap_authors':len(set(df_ea['author_hash'])&set(df_post['author_hash'])),
        'pct_edited':round(100*df_game['was_edited'].mean(),1),
        # before the release date but flagged post launch, should be zero
        'n_flag_date_disagree':int(((~df_game['written_during_early_access'])&
                                    (df_game['review_date']<df_game['release_date'])).sum())})

df_summary=pd.DataFrame(summary_rows)
if df_summary.empty:
    raise SystemExit('no clean games on disk yet - re run once the collector has produced data')

print('PER GAME VERIFICATION')
print(df_summary.head(15).to_string(index=False))
print(f'... showing 15 of {len(df_summary)} games')

PER GAME VERIFICATION
  appid release_date  n_reviews  n_early_access  n_post_launch  pct_with_playtime  pct_persistence_negative  median_persistence_hours  n_ea_authors  n_overlap_authors  pct_edited  n_flag_date_disagree
1000030   2020-01-29       1735             620           1115              100.0                       0.1                      27.7           620                  0        10.2                     0
1000080   2019-01-18        154              21            133              100.0                       0.0                       0.6            21                  0        20.1                     0
1000360   2021-02-16         43              43              0              100.0                       0.0                       0.7            43                  0         9.3                     0
1004270   2019-01-30        577             508             69              100.0                       0.2                       1.8           508                  0        

In [7]:
# what the english filter cost per game, using the denominator counted earlier
if not rejects_file_path.exists():
    raise SystemExit('no cleaning has run yet - nothing to report on. re run once raw data exists')
df_rejects_now=pd.read_csv(rejects_file_path)
df_language=df_rejects_now[['appid','n_reviews_all_languages','n_english','english_share',
                            'top_other_language','not_latin_script','empty_after_cleaning','kept']]

print('WHAT THE ENGLISH FILTER COST, PER GAME')
print(df_language.head(15).to_string(index=False))
print(f'... showing 15 of {len(df_language)} games')
print(f'reviews collected across all languages : {df_language["n_reviews_all_languages"].sum():,}')
print(f'reviews kept after english filter      : {df_language["kept"].sum():,}')
print(f'overall english share                  : '
      f'{100*df_language["n_english"].sum()/df_language["n_reviews_all_languages"].sum():.1f}%')
print(f'english share range across games       : '
      f'{df_language["english_share"].min()}% to {df_language["english_share"].max()}%')
print(f'claimed english but not latin script   : {df_language["not_latin_script"].sum():,}')
print(f'nothing left after cleaning            : {df_language["empty_after_cleaning"].sum():,}')
print('that range is the reason english_share is carried forward as a per game covariate.')
print('a game measured on 22% of its community is not measured the same way as one at 72%')

df_language.to_csv(tables_dir/'english_filter_effect.csv',index=False)
print('written to',tables_dir/'english_filter_effect.csv')

WHAT THE ENGLISH FILTER COST, PER GAME
  appid  n_reviews_all_languages  n_english  english_share top_other_language  not_latin_script  empty_after_cleaning  kept
1000080                     1571        158          10.06           schinese                 4                     0   154
1000360                       77         44          57.14           schinese                 1                     0    43
1004270                     1731        577          33.33           schinese                 0                     0   577
1005400                      258        173          67.05             french                 0                     0   173
1005930                     1627        213          13.09            russian                14                     0   199
1010860                      748        115          15.37            turkish                 2                     1   112
1011290                      212        185          87.26            russian                

In [8]:
print('DO MY DESIGN ASSUMPTIONS HOLD')

# assumption one, the early access flag splits the corpus
games_with_both_windows=(df_summary['n_early_access']>0)&(df_summary['n_post_launch']>0)
print(f'games with both an early access and a post launch window : '
      f'{games_with_both_windows.sum()} of {len(df_summary)}')

# assumption two, the playtime fields are populated
print(f'median share of reviews carrying playtime_at_review      : '
      f'{df_summary["pct_with_playtime"].median():.1f}%')

# assumption three, persistence computable on fifteen of twenty games
games_with_persistence=df_summary['median_persistence_hours'].notna()&(df_summary['n_ea_authors']>=200)
print(f'games with >=200 early access reviews and computable persistence : '
      f'{games_with_persistence.sum()} of {len(df_summary)}')

# assumption four, how noisy is the metric
print(f'median rate of negative persistence (steam recalculated playtime): '
      f'{df_summary["pct_persistence_negative"].median():.1f}%')

# and the thing rq2 needs, are there enough authors to exclude
print(f'median early access authors also reviewing post launch    : '
      f'{df_summary["n_overlap_authors"].median():.0f}')

print('total clean reviews so far:',f'{df_summary["n_reviews"].sum():,}')

DO MY DESIGN ASSUMPTIONS HOLD
games with both an early access and a post launch window : 1223 of 1793
median share of reviews carrying playtime_at_review      : 100.0%
games with >=200 early access reviews and computable persistence : 1031 of 1793
median rate of negative persistence (steam recalculated playtime): 0.0%
median early access authors also reviewing post launch    : 0
total clean reviews so far: 1,865,234


# Two things the verification caught that change the design

The store release date is not the launch date, and the early-access flag is recomputed when a review is edited. Both change how the cohort boundary is derived.

In [9]:
print('FINDING 1 : IS app_release_date THE FULL RELEASE DATE OR THE EARLY ACCESS DATE')

# if the store date were the launch date, both of these checks would be clean
for _,game in df_summary.head(15).iterrows():
    df_game=pd.read_parquet(clean_reviews_dir/f'{game["appid"]}.parquet',
                            columns=['written_during_early_access','review_date','release_date'])
    ea_after_release=((df_game['written_during_early_access'])&
                      (df_game['review_date']>df_game['release_date'])).sum()
    verdict='EA START DATE' if ea_after_release>0.5*len(df_game) else 'looks like full release'
    print(f'app {game["appid"]:>8} | steam says {game["release_date"]} | '
          f'{ea_after_release:>6,} EA reviews dated AFTER it | {verdict}')

print()
print(f'... showing 15 of {len(df_summary)} games')
print()
print('an early access review cannot postdate the full release, so a large count in that')
print('column means the date is the early access start and the collection windows built')
print('from it are measuring the wrong period')

print()
print('FINDING 2 : THE EARLY ACCESS FLAG MOVES WHEN A REVIEW IS EDITED')
print(f'median share of reviews that were edited after posting : {df_summary["pct_edited"].median():.1f}%')
print(f'total reviews where the flag and the date disagree     : {df_summary["n_flag_date_disagree"].sum():,}')
print()
print('so the flag describes the current text, not the moment of posting. i keep both')
print('signals and report the disagreement rather than choosing one silently')

FINDING 1 : IS app_release_date THE FULL RELEASE DATE OR THE EARLY ACCESS DATE
app  1000030 | steam says 2020-01-29 |    620 EA reviews dated AFTER it | looks like full release
app  1000080 | steam says 2019-01-18 |     21 EA reviews dated AFTER it | looks like full release
app  1000360 | steam says 2021-02-16 |     43 EA reviews dated AFTER it | EA START DATE
app  1004270 | steam says 2019-01-30 |    508 EA reviews dated AFTER it | EA START DATE
app  1005400 | steam says 2020-10-30 |     69 EA reviews dated AFTER it | looks like full release
app  1005930 | steam says 2019-01-19 |    199 EA reviews dated AFTER it | EA START DATE
app  1010860 | steam says 2019-02-01 |     22 EA reviews dated AFTER it | looks like full release
app  1011290 | steam says 2019-08-29 |     23 EA reviews dated AFTER it | looks like full release
app  1012110 | steam says 2022-12-16 |  1,280 EA reviews dated AFTER it | EA START DATE
app  1013320 | steam says 2024-04-01 |      0 EA reviews dated AFTER it | looks

# The author overlap table, which is a result rather than a check

Overlap is zero on every game: Steam allows one review per account per game, so the two cohorts cannot share an author.

In [10]:
# a report result, so write it where the figures notebook can find it
df_overlap=df_summary[['appid','n_early_access','n_post_launch','n_ea_authors','n_overlap_authors']].copy()
# zero denominator, `where` keeps the column numeric
post_launch_authors=df_overlap['n_post_launch'].where(df_overlap['n_post_launch']>0)
df_overlap['pct_of_post_launch_excluded']=(100*df_overlap['n_overlap_authors']/post_launch_authors).round(2)
df_overlap.to_csv(tables_dir/'author_overlap.csv',index=False)

print('RQ2 COHORT EXCLUSION : HOW MANY AUTHORS DOES IT ACTUALLY REMOVE')
print(df_overlap.head(15).to_string(index=False))
print(f'... showing 15 of {len(df_overlap)} games')
print(f'games checked                      : {len(df_overlap)}')
print(f'reviews behind this table          : {df_summary["n_reviews"].sum():,}')
print(f'total authors the exclusion removes: {df_overlap["n_overlap_authors"].sum()}')
print('so the control is a no op. one review per account per game means the two sets')
print('cannot intersect. i report the measured zero and narrow the claim to instrumental')
print('circularity rather than pretending a same person control was applied')
print('written to',tables_dir/'author_overlap.csv')

RQ2 COHORT EXCLUSION : HOW MANY AUTHORS DOES IT ACTUALLY REMOVE
  appid  n_early_access  n_post_launch  n_ea_authors  n_overlap_authors  pct_of_post_launch_excluded
1000030             620           1115           620                  0                          0.0
1000080              21            133            21                  0                          0.0
1000360              43              0            43                  0                          NaN
1004270             508             69           508                  0                          0.0
1005400              69            104            69                  0                          0.0
1005930             199              0           199                  0                          NaN
1010860              22             90            22                  0                          0.0
1011290              23            160            23                  0                          0.0
1012110            1280    

# The assertion cell

In [11]:
required_columns={'review_id','appid','author_hash','playtime_at_review','playtime_forever',
                  'post_review_persistence','written_during_early_access','review_date',
                  'review_text_raw','review_text_clean','window','has_playtime','was_edited',
                  'timestamp_updated'}

clean_files=sorted(clean_reviews_dir.glob('*.parquet'))
assert len(clean_files)>0,'no clean parquet files were written'

for clean_file_path in clean_files:
    df_check=pd.read_parquet(clean_file_path)
    assert not df_check.empty,f'{clean_file_path.name} is empty'
    assert required_columns.issubset(df_check.columns),f'{clean_file_path.name} is missing columns'
    assert df_check['review_id'].is_unique,f'{clean_file_path.name} still has duplicate review ids'
    assert df_check['author_hash'].str.len().eq(16).all(),f'{clean_file_path.name} has a bad author hash'
    assert df_check['appid'].nunique()==1,f'{clean_file_path.name} mixes more than one game'

    # the corpus is english and the cleaned column is never empty
    assert (df_check['language']=='english').all(),f'{clean_file_path.name} kept a non english review'
    assert (df_check['review_text_clean'].str.strip()!='').all(),f'{clean_file_path.name} kept an empty cleaned review'
    # the cleaned column has no markup, urls or hearts left
    assert not df_check['review_text_clean'].str.contains('\u2665').any(),f'{clean_file_path.name} left hearts in the clean text'
    # a functional url, not the bare word
    assert not df_check['review_text_clean'].str.contains(r'https?://\S',regex=True).any(),        f'{clean_file_path.name} left a working url in the clean text'

    # the carve-out holds both ways
    was_censored=df_check['review_text_raw'].str.contains('\u2665\u2665')
    if was_censored.any():
        assert df_check.loc[was_censored,'review_text_clean'].str.contains(r'\[censored\]',regex=True).all(),\
            f'{clean_file_path.name} dropped the censorship marker instead of tokenising it'

# the hash must be stable across runs or the rq2 exclusion breaks
assert hash_steam_id('76561198137153398')==hash_steam_id('76561198137153398')
assert len(hash_steam_id('76561198137153398'))==16

print(f'all {len(clean_files)} clean files passed')
print(f'clean review layer holds {df_summary["n_reviews"].sum():,} reviews across {len(clean_files)} games')

all 1793 clean files passed
clean review layer holds 1,865,234 reviews across 1793 games
